# 1. Middleware概述

中间件是插入在 Agent 执行流程中的通用处理层，用来在模型调用、工具调用、状态更新等关键节点前后进行拦截、增强和控制。    
中间件不直接完成业务任务，而是为执行过程提供通用能力。

| 目标 | 典型能力 |
| --- | --- |
| 稳定性 | 超时、重试、降级 |
| 可观测性 | 日志、Tracing、指标统计 |
| 安全性 | 权限校验、敏感信息过滤 |
| 成本控制 | 限流、Token 预算、上下文裁剪 |
| 输出质量 | 结构校验、格式转换、后处理 |

# 2. 常用内置中间件

## 2.1 SummarizationMiddleware
+ 作用： 对历史消息进行摘要&总结，达到压缩上下文的目的
+ 原理： 自定义触发条件，达到触发条件时，调用大模型对历史信息进行摘要，将摘要内容作为**HumanMessage**放到消息列表最开始的位置

**重点参数说明：**

| 参数 | 作用 | 常用写法 |
| --- | --- | --- |
| model | 用于生成摘要的模型 | 可使用成本更低、速度更快的模型 |
| trigger | 何时触发摘要 | ("messages", 6)、("tokens", 3000)、("fraction", 0.8) |
| keep | 摘要后保留多少最近消息 | ("messages", 1)、("tokens", 1000) |
| summary_prompt | 自定义摘要提示词 | 强调保留关键信息、事实、偏好和未完成任务 |

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from rich import print as rprint

load_dotenv(".env")

# 会话总结模型
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    profile={"max_input_tokens": 128_000},
)
rprint(model.profile)
# rprint(init_chat_model(model="deepseek-v4-pro").profile)

agent = create_agent(
    model="deepseek-v4-pro",
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[("tokens", 100), ("messages", 6), ("fraction", 0.01)],
            keep=(
                "messages",
                1,
            ),  # trigger 可以写成列表，表示多个触发条件；但 keep 只能是一个二元组
        )
    ],
)

messages = [
    SystemMessage(
        "你是一个幽默风趣、知识渊博的古风AI书生，擅长用文言与白话夹杂的方式解答问题。"
    ),
    HumanMessage("在下路过此地，见你端坐松下，手持竹简，敢问阁下是何方高人？"),
    AIMessage(
        "哈哈，山野闲人，姓柳名砚，字墨卿——人称‘半卷诗书半壶酒’。阁下既来，可愿共饮一盏，论道谈天？"
    ),
    HumanMessage("好啊！那我先考考你：‘床前明月光’下一句是？"),
    AIMessage(
        "举头望明月，低头思故乡～不过嘛……（轻摇折扇）若按老夫新编版：‘举头望明月，低头摸手机——信号满格却无网！’"
    ),
    HumanMessage("哈哈哈！你这书生，倒是个现代隐士！"),
    AIMessage(
        "非也非也～老夫乃‘数字山林派’开山祖师，左手《论语》，右手Python，吟诗作对还能debug～"
    ),
    HumanMessage("那……能教我写个‘自动给老板回邮件’的脚本吗？"),
]

resp = agent.invoke({"messages": messages})
for msg in resp["messages"]:
    msg.pretty_print()

## 2.2 HumanInTheLoopMiddleware
+ 作用： 在关键工具执行前暂停 Agent，交由人工审批、修改、拒绝或补充回复
+ 原理： 为指定工具配置中断规则，当 Agent 准备调用这些工具时先生成 interrupt，等待人工决策后再继续执行

**重点参数说明：**

| 参数 | 作用 | 常用写法 |
| --- | --- | --- |
| interrupt_on | 指定哪些工具需要人工介入 | {"send_email": True} |
| True | 允许人工批准、修改、拒绝或直接回复 | {"delete_file": True} |
| False | 自动通过，不触发中断 | {"search": False} |
| InterruptOnConfig | 精细控制允许的人工决策和触发条件 | 为高风险工具设置审批策略 |
| description_prefix | 设置中断提示的前缀说明 | "该工具调用需要人工确认" |

## 2.3 PIIMiddleware
+ 作用： 检测并处理输入、输出或工具结果中的敏感信息，降低隐私泄露风险
+ 原理： 根据指定的 PII 类型匹配文本内容，命中后按策略进行阻断、脱敏、掩码或哈希处理

**重点参数说明：**

| 参数 | 作用 | 常用写法 |
| --- | --- | --- |
| pii_type | 指定要检测的敏感信息类型 | "email"、"credit_card"、"ip"、"url" |
| strategy | 命中敏感信息后的处理方式 | "redact"、"mask"、"hash"、"block" |
| detector | 自定义检测规则 | 传入正则表达式或检测函数 |
| apply_to_input | 是否检查用户输入 | 默认 True |
| apply_to_output | 是否检查模型输出 | 需要保护返回内容时设为 True |
| apply_to_tool_results | 是否检查工具返回结果 | 工具可能返回敏感数据时设为 True |

**strategy 参数详解：**

| 策略 | 作用 | 示例 |
| --- | --- | --- |
| redact | 用固定占位符完全替换敏感内容，无法还原 | "zhangsan@example.com" → "[REDACTED]" |
| mask | 部分遮蔽敏感信息，保留格式轮廓，便于人工识别 | "zhangsan@example.com" → "zha****@example.com" |
| hash | 对敏感内容做哈希处理，便于后续去重或关联分析而不泄露原文 | "zhangsan@example.com" → "a1b2c3d4..." |
| block | 检测到敏感信息时直接阻断请求，返回错误，不继续执行 | 用户输入含敏感信息 → 请求被拒绝 |

**案例：使用内置检测器** 

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from rich import print as rprint

load_dotenv(".env")

model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware(pii_type="email", strategy="redact", apply_to_input=True),
        PIIMiddleware(pii_type="credit_card", strategy="mask", apply_to_input=True),
        PIIMiddleware(pii_type="mac_address", strategy="mask", apply_to_input=True),
        PIIMiddleware(pii_type="url", strategy="hash", apply_to_input=True),
    ],
)

human_massages = HumanMessage(
    content=(
        "帮我在以下任务中执行操作：\n"
        "- 向 156168188@qq.com 发送一封邮件；\n"
        "- 查询银行卡号 5105-1051-0510-5100 的余额；\n"
        "- 访问 https://localhost:12345；\n"
        "- 确认地址 11-11-11-11-11-11 是否为 MAC 地址。"
    )
)


resp = agent.invoke({"messages": [human_massages]})

for msg in resp["messages"]:
    msg.pretty_print()

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from rich import print as rprint

load_dotenv(".env")

model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware(pii_type="ip", strategy="block", apply_to_input=True),
    ],
)

resp = agent.invoke(
    {"messages": [HumanMessage("帮我看看 19.2.168.10.1 这个ip能不能ping通")]}
)

for msg in resp["messages"]:
    msg.pretty_print()

**案例：自定义手机号检测器**

内置 PII 类型不包含手机号时，可以通过 `detector` 传入自定义检测函数。检测函数接收文本，返回匹配到的敏感信息类型、原始值和位置。

In [ ]:
import re
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.messages import HumanMessage


def detect_phone(content: str) -> list[dict]:
    pattern = r"(?<!\d)1[3-9]\d{9}(?!\d)"
    return [
        {
            "type": "phone",
            "value": match.group(),
            "start": match.start(),
            "end": match.end(),
        }
        for match in re.finditer(pattern, content)
    ]


agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware(
            pii_type="phone",
            detector=detect_phone,
            strategy="mask",
            apply_to_input=True,
        )
    ],
)

resp = agent.invoke(
    {
        "messages": [
            HumanMessage("我的手机号是 13812345678")
        ]
    }
)

for msg in resp["messages"]:
    msg.pretty_print()

## 2.4 TodoListMiddleware
+ 作用： 在执行过程中自动维护一个结构化的待办事项列表，帮助 Agent 对复杂任务进行分解、规划和追踪
+ 原理： 拦截模型输出，从中提取或更新 todo 列表，将当前任务进度以结构化形式注入到上下文中，使 Agent 能够清晰地了解已完成、进行中和待处理的任务

**重点参数说明：**

| 参数 | 作用 | 常用写法 |
| --- | --- | --- |
| system_prompt | 自定义系统提示词，指导模型何时及如何使用任务列表 | 不传则使用内置默认提示词 |
| tool_description | 自定义 write_todos 工具的描述文本 | 不传则使用内置默认描述 |

TodoListMiddleware 提供两个内置工具：write_todos（创建/全量更新任务列表）和 read_todos（读取当前任务状态）。在 create_deep_agent() 中默认自动包含，使用 create_agent() 时需显式挂载。

**TodoListMiddleware vs LangGraph：**

| | TodoListMiddleware | LangGraph |
| --- | --- | --- |
| 定位 | Agent 内的任务规划插件 | 独立的流程编排框架 |
| 控制流 | 由模型自主决策，无显式控制 | 开发者显式定义节点、条件、循环 |
| 复杂度 | 轻量，一行配置即可启用 | 需定义 State、Graph、Edge |
| 多 Agent | 不涉及，单 Agent 内使用 | 天然支持多 Agent / 多节点协作 |
| 状态持久化 | 依赖 Agent 上下文，无独立持久化 | 内置 Checkpointer，支持断点续跑 |

**选型建议：**

| 场景 | 推荐 |
| --- | --- |
| 单 Agent 处理"步骤较多但逻辑线性"的任务（如调研报告、多步分析） | TodoListMiddleware |
| 需要条件分支、循环重试、并行执行等自定义流程控制 | LangGraph |
| 希望快速给 Agent 加个"任务清单"能力，不想额外设计流程 | TodoListMiddleware |
| 构建生产级多 Agent 流水线，需要状态持久化、人工审核节点 | LangGraph |

 **一句话总结**：TodoListMiddleware 让 Agent **自己**规划步骤；LangGraph 让开发者**为** Agent 规划步骤。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain_core.messages import HumanMessage, SystemMessage
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(".env")

model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)

agent = create_agent(
    model=model,
    middleware=[TodoListMiddleware()],
)

resp = agent.invoke(
    {
        "messages": [
            HumanMessage(
                "请帮我完成一个新项目的启动准备工作：\n"
                "1）项目背景：我们要做一个面向程序员的知识库Web应用，请先为这个产品起一个候选名称并做SWOT分析；\n"
                "2）技术选型：对比推荐前后端技术栈并给出推荐方案；\n"
                "3）数据库设计和API设计\n"
                "4）部署方案：给出生产环境的部署架构建议。\n\n"
                "请先做好整体规划再逐一完成。"
            )
        ]
    }
)

rprint(resp)

# 3. 其他内置中间件

## 3.1 ModelCallLimitMiddleware
+ 作用： 限制模型调用次数，防止 Agent 陷入无限循环或单次运行消耗过多 Token，控制成本与执行时长
+ 原理： 在每次模型调用前检查计数器，达到阈值后按配置行为终止运行（优雅结束或抛出错误），调用后递增计数器；run_limit 每次 invoke 重置，thread_limit 跨会话持久化

**重点参数说明：**

| 参数 | 作用 | 常用写法 |
| --- | --- | --- |
| run_limit | 单次 invoke 内最大模型调用次数，调用结束自动重置 | run_limit=10 |
| thread_limit | 整个会话线程内最大模型调用次数，需配合 checkpointer 持久化 | thread_limit=50 |
| exit_behavior | 达到限制后的行为 | "end"（优雅终止，插入提示消息）、"error"（抛出异常） |

**示例准备：使用 Fake 模型稳定复现限制行为**

`run_limit=1` 需要让一次 `invoke` 内发生第二次模型调用，因此示例中让模型先调用工具；工具返回后，Agent 准备第二次调用模型时触发限制。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain.agents.middleware.model_call_limit import ModelCallLimitExceededError
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver


class BindableFakeMessagesListChatModel(FakeMessagesListChatModel):
    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        return self


@tool
def ping() -> str:
    """Return pong."""
    return "pong"

**案例 1：`run_limit` + `end`**

单次运行最多允许 1 次模型调用。第一次模型调用发起工具调用，工具执行后准备第二次模型调用时，`end` 会优雅终止并插入一条提示消息。

In [ ]:
model = BindableFakeMessagesListChatModel(
    responses=[
        AIMessage(content="", tool_calls=[{"name": "ping", "args": {}, "id": "call_1"}]),
        AIMessage(content="工具调用完成"),
    ]
)

agent = create_agent(
    model=model,
    tools=[ping],
    middleware=[ModelCallLimitMiddleware(run_limit=1, exit_behavior="end")],
)

resp = agent.invoke({"messages": [HumanMessage("请调用 ping 工具")]})
rprint(resp)

**案例 2：`run_limit` + `error`**

触发位置与案例 1 相同，但 `error` 会抛出 `ModelCallLimitExceededError`。这里捕获异常，保证单元格可以正常跑完。

In [ ]:
model = BindableFakeMessagesListChatModel(
    responses=[
        AIMessage(content="", tool_calls=[{"name": "ping", "args": {}, "id": "call_1"}]),
        AIMessage(content="工具调用完成"),
    ]
)

agent = create_agent(
    model=model,
    tools=[ping],
    middleware=[ModelCallLimitMiddleware(run_limit=1, exit_behavior="error")],
)

try:
    resp = agent.invoke({"messages": [HumanMessage("请调用 ping 工具")]})
    rprint(resp)
except ModelCallLimitExceededError as e:
    print(type(e).__name__, "|", e)

**案例 3：`thread_limit` + `end`**

`thread_limit` 需要配合 `checkpointer` 和相同的 `thread_id`。第一次 `invoke` 用掉 1 次模型调用，第二次同线程调用时触发限制并优雅终止。

In [ ]:
checkpointer = InMemorySaver()
model = BindableFakeMessagesListChatModel(
    responses=[
        AIMessage(content="第一次模型回答"),
        AIMessage(content="第二次模型回答"),
    ]
)

agent = create_agent(
    model=model,
    middleware=[ModelCallLimitMiddleware(thread_limit=1, exit_behavior="end")],
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "thread-limit-end-demo"}}

resp1 = agent.invoke({"messages": [HumanMessage("第一次调用")]}, config=config)
resp2 = agent.invoke({"messages": [HumanMessage("第二次调用")]}, config=config)

print("第一次 invoke：")
rprint(resp1)
print("\n第二次 invoke：")
rprint(resp2)

**案例 4：`thread_limit` + `error`**

同样使用相同 `thread_id` 连续调用两次。第二次调用超过线程级限制，直接抛出异常。

In [ ]:
checkpointer = InMemorySaver()
model = BindableFakeMessagesListChatModel(
    responses=[
        AIMessage(content="第一次模型回答"),
        AIMessage(content="第二次模型回答"),
    ]
)

agent = create_agent(
    model=model,
    middleware=[ModelCallLimitMiddleware(thread_limit=1, exit_behavior="error")],
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "thread-limit-error-demo"}}

agent.invoke({"messages": [HumanMessage("第一次调用")]}, config=config)

try:
    resp = agent.invoke({"messages": [HumanMessage("第二次调用")]}, config=config)
    rprint(resp)
except ModelCallLimitExceededError as e:
    print(type(e).__name__, "|", e)

## 3.2 ToolCallLimitMiddleware
+ 作用： 限制工具调用次数，防止 Agent 反复调用高成本、高风险或不应频繁使用的工具
+ 原理： 在工具执行前检查调用计数，达到阈值后按配置行为处理；可限制全部工具，也可只限制指定工具

**重点参数说明：**

| 参数 | 作用 | 常用写法 |
| --- | --- | --- |
| `tool_name` | 指定要限制的工具名称；为 `None` 时限制所有工具 | `tool_name="search"` |
| `run_limit` | 单次 `invoke` 内最大工具调用次数 | `run_limit=3` |
| `thread_limit` | 整个会话线程内最大工具调用次数，需配合 `checkpointer` | `thread_limit=10` |
| `exit_behavior` | 达到限制后的处理方式 | `"continue"`、`"end"`、`"error"` |
| `continue` | 阻止超限工具调用，返回工具错误消息，让模型继续决策 | 默认值，适合多数场景 |
| `end` | 立即终止 Agent 执行 | 适合超限后不希望继续推理的场景 |
| `error` | 直接抛出异常 | 适合测试、监控和外层统一错误处理 |

**使用建议：**

- 限制高成本工具：如搜索、代码执行、数据库查询。
- 限制高风险工具：如发邮件、删文件、提交订单。
- `run_limit` 控制单次任务成本，`thread_limit` 控制同一会话的累计使用量。
- 同时设置 `run_limit` 和 `thread_limit` 时，`run_limit` 不能大于 `thread_limit`。

**案例：限制自定义工具在单次运行中只能调用 1 次**

下面示例使用真实模型和自定义工具 `repeat_marker`。第一次工具调用正常执行，工具结果会要求模型再次调用同一工具；第二次调用超过 `run_limit=1`，中间件返回工具错误消息。由于 `exit_behavior="continue"`，Agent 会继续执行并给出最终回答。

In [ ]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool


load_dotenv(".env")

model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)


@tool
def repeat_marker(step: str) -> str:
    """Record a demo step and ask the agent to call this tool again when step is first."""
    if step == "first":
        return "第一次工具调用成功。请继续调用 repeat_marker，参数 step='second'。"
    return "第二次工具调用成功。"


agent = create_agent(
    model=model,
    tools=[repeat_marker],
    middleware=[
        ToolCallLimitMiddleware(
            tool_name="repeat_marker",
            run_limit=1,
            exit_behavior="continue",
        )
    ],
)

resp = agent.invoke(
    {
        "messages": [
            HumanMessage(
                "请严格按步骤执行：\n"
                "1. 先调用 repeat_marker，参数 step='first'。\n"
                "2. 根据工具返回结果，再调用一次 repeat_marker，参数 step='second'。\n"
                "3. 如果第二次工具调用被限制，请说明限制已经生效。"
            )
        ]
    }
)
for msg in resp["messages"]:
    print(type(msg).__name__, "|", msg.content)

## 3.3 ModelFallbackMiddleware
+ 作用： 当主模型调用失败时，自动切换到备用模型，提高 Agent 的可用性
+ 原理： `create_agent(model=...)` 中的模型先执行；如果主模型抛出异常，中间件会按顺序尝试备用模型，直到成功或全部失败

**重点参数说明：**

| 参数 | 作用 | 常用写法 |
| --- | --- | --- |
| `first_model` | 第一个备用模型 | `ModelFallbackMiddleware(deepseek_model)` |
| `additional_models` | 后续备用模型，按顺序尝试 | `ModelFallbackMiddleware(model_a, model_b)` |
| 主模型 | 写在 `create_agent(model=...)` 中 | `create_agent(model=qwen_model, ...)` |
| 触发条件 | 主模型调用抛出异常 | 网络错误、限流、认证失败、服务不可用等 |

注意：模型回答质量不佳不会触发 fallback，只有模型调用过程抛出异常才会触发。

**案例：主模型使用千问，备用模型使用 DeepSeek**

如果千问主模型调用成功，会直接返回主模型结果；如果主模型调用失败，`ModelFallbackMiddleware` 会自动切换到 DeepSeek 备用模型。

In [ ]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import ModelFallbackMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage


load_dotenv(".env")

# 主模型：千问
qwen_model = init_chat_model(
    model="qwen3.7-max", 
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY_FAKE")
)

# 备用模型：DeepSeek
deepseek_model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)

agent = create_agent(
    model=qwen_model,
    middleware=[ModelFallbackMiddleware(deepseek_model)],
)

resp = agent.invoke(
    {
        "messages": [
            HumanMessage("请介绍一下自己")
        ]
    }
)

for msg in resp["messages"]:
    msg.pretty_print()

**验证 fallback 的方法：**

正常情况下，如果千问主模型可用，就不会进入 DeepSeek。若要验证备用模型是否生效，可以临时使用错误的 DashScope Key，或一个不可用的千问模型名称，让主模型调用失败。

## 3.4 LLMToolSelectorMiddleware
+ 作用： 当 Agent 可用工具较多时，先用一个 LLM 从工具列表中筛选出最相关的工具，再交给主模型使用
+ 原理： 在主模型调用前，选择模型根据用户问题和工具描述输出候选工具名；中间件据此裁剪工具列表，减少无关工具干扰和 token 消耗
+ 最佳方案： Embedding 语义检索替代 LLMToolSelectorMiddleware LLM 选择器

**重点参数说明：**

| 参数 | 作用 | 常用写法 |
| --- | --- | --- |
| `model` | 用于选择工具的模型；不传则使用 Agent 主模型 | `model=qwen_model` |
| `system_prompt` | 自定义工具选择规则 | 强调按用户问题选择最相关工具 |
| `max_tools` | 最多选择几个工具 | `max_tools=2` |
| `always_include` | 始终保留的工具名列表，不参与筛选 | `always_include=["final_check"]` |

**使用建议：**

- 工具数量较少时通常不需要使用。
- 工具很多、描述较长、领域差异明显时更适合使用。
- 可以用更便宜、更快的模型做工具选择，用更强的模型做最终推理。
- `always_include` 适合放审计、兜底、安全检查等必须保留的工具。

**案例：从多个自定义工具中选择天气工具**

下面示例提供 3 个工具，但用户只询问天气。`LLMToolSelectorMiddleware(max_tools=1)` 会先筛选出最相关的工具，再让主模型继续执行。

In [ ]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolSelectorMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

load_dotenv(".env")


@tool
def get_weather(city: str) -> str:
    """查询指定城市的实时天气信息。

    当用户询问以下内容时调用此工具：
    - 某城市的天气（"北京天气"、"上海今天多少度"、"杭州热不热"）
    - 是否会下雨/下雪（"明天要带伞吗"、"深圳会下雨吗"）

    以下情况不应调用此工具：
    - 询问气候特点（"北京冬天冷吗"）→ 常识问题，直接回答
    - 询问空气质量、自然灾害 → 需用其他专用工具
    - 询问历史天气 → 本工具仅支持实时天气

    返回字符串："{城市} 今天{天气}，{温度}度"
    """
    print("调用工具：get_weather\n")
    return f"{city} 今天晴，26 度。"


@tool
def calculate_tax(amount: float) -> str:
    """计算指定金额对应的税费。

    当用户询问以下内容时调用此工具：
    - 金额的税费计算（"100万的房子税费多少"、"这笔收入要交多少税"）
    - 个人所得税/增值税/契税等具体税种计算（"个人所得税怎么算"）

    以下情况不应调用此工具：
    - 汇率换算（不在本工具范围）→ 需用汇率工具
    - 折扣/优惠计算 → 非税费类计算
    - 房贷利息、投资回报 → 非税费类计算

    返回字符串：含税费金额和计算公式
    """
    print("调用工具：calculate_tax\n")
    return f"税费为 {amount * 0.06:.2f}"


@tool
def search_docs(query: str) -> str:
    """搜索课程文档库，返回匹配的文档标题和摘要。

    当用户询问以下内容时调用此工具：
    - 课程/教程/文档相关内容（"Python入门文档"、"LangChain教程"、"怎么配置中间件"）
    - 技术学习资料（"有没有关于Agent开发的资料"）

    以下情况不应调用此工具：
    - 搜索互联网/外部网页 → 非课程文档范围
    - 查询天气、计算税费、汇率换算 → 非文档类问题
    - 执行代码或调试程序 → 需用代码执行工具

    返回字符串：匹配到的文档数量和标题摘要
    """
    print("调用工具：search_docs\n")
    return "找到 3 篇相关文档。"


# 从工具对象动态提取名称，避免硬编码导致的幻觉问题
# 否则 selector 模型可能把 search_docs 翻译成 search_documents
tools = [get_weather, calculate_tax, search_docs]
tool_names = "、".join(t.name for t in tools)

tool_selector_prompt = (
    "你的任务是根据用户问题选择最相关的工具。\n"
    f"可用工具名只有：{tool_names}。\n"
    "必须从已有的工具名中选择，不允许创造、翻译、改写工具名。\n"
    '请严格返回 JSON object，格式为：{"tools": ["工具名"]}。'
)

qwen_model = init_chat_model(
    model="qwen3.7-max",
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    extra_body={"enable_thinking": False},
)

agent = create_agent(
    model=qwen_model,
    tools=tools,
    middleware=[
        LLMToolSelectorMiddleware(
            model=qwen_model,
            max_tools=3,  # 改为 3 后可以同时调用 search_docs  get_weather  calculate_tax
            system_prompt=tool_selector_prompt,
        )
    ],
)

resp = agent.invoke(
    {"messages": [HumanMessage("帮我查一下文档，顺便查下北京的天气，最后查一下税费")]}
)
rprint(resp)

报错 1："<400> InternalError.Algo.InvalidParameter: 'messages' must contain the word 'json' in some form, to use 'response_format' of type 'json_object'."

**原因**：LLMToolSelectorMiddleware 内部会强制使用结构化输出，要求模型返回: {"tools": ["tool_name"]}
它的工作流程是：用户输入问题 -> LLMToolSelectorMiddleware 先调用一个模型，让它从工具列表里选工具 -> 为了拿到工具名，它内部执行了：structured_model = qwen_model.with_structured_output(schema)，也就是它要求模型返回类似: {"tools": ["tool_name"]}

'messages' must contain the word 'json' in some form, to use 'response_format' of type 'json_object'.", 'type': 'invalid_request_error'
意思是：如果请求里用了 response_format={"type": "json_object"}，那么 prompt 里必须明确出现 json 这个词  

**解决方案**：给 LLMToolSelectorMiddleware 加一个包含 JSON 的 system_prompt

tool_selector_prompt = (
    "你的任务是根据用户问题选择最相关的工具。"
    "请严格返回 JSON object，格式为：{\"tools\": [\"工具名\"]}。"
    "只能返回可用工具名称，不要返回解释。"
)

报错 2：'Thinking mode does not support this tool_choice'

**原因**：qwen3.7-max 默认开启 thinking mode，而 thinking mode 不支持当前这次请求里的 tool_choice  

**解决方案**：关闭模型的think模式：extra_body={"enable_thinking": False}

报错 3 ：ValueError: Model selected invalid tools: ['get_current_weather']  

**原因**：LLMToolSelectorMiddleware 的 selector 模型（qwen3.7-max）返回了不存在的工具名 get_current_weather，模型的幻觉问题  

**解决方案**：为 selector 模型的 prompt 中明确限定只能返回真实工具名。prompt add  "可用工具名只有：get_weather、calculate_tax、search_docs。必须从已有的工具名中选择，不允许创造、翻译、改写工具名。"

In [ ]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolSelectorMiddleware
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from rich import print as rprint

load_dotenv(".env")

@tool
def get_weather(city: str) -> str:
    """查询指定城市的实时天气信息。

    当用户询问以下内容时调用此工具：
    - 某城市的天气（"北京天气"、"上海今天多少度"、"杭州热不热"）
    - 是否会下雨/下雪（"明天要带伞吗"、"深圳会下雨吗"）

    以下情况不应调用此工具：
    - 询问气候特点（"北京冬天冷吗"）→ 常识问题，直接回答
    - 询问空气质量、自然灾害 → 需用其他专用工具
    - 询问历史天气 → 本工具仅支持实时天气

    返回字符串："{城市} 今天{天气}，{温度}度"
    """
    print("调用工具：get_weather")
    return f"{city} 今天晴，26 度。"


@tool
def calculate_tax(amount: float) -> str:
    """计算指定金额对应的税费。

    当用户询问以下内容时调用此工具：
    - 金额的税费计算（"100万的房子税费多少"、"这笔收入要交多少税"）
    - 个人所得税/增值税/契税等具体税种计算（"个人所得税怎么算"）

    以下情况不应调用此工具：
    - 汇率换算（不在本工具范围）→ 需用汇率工具
    - 折扣/优惠计算 → 非税费类计算
    - 房贷利息、投资回报 → 非税费类计算

    返回字符串：含税费金额和计算公式
    """
    print("调用工具：calculate_tax")
    return f"税费为 {amount * 0.06:.2f}"


@tool
def search_docs(query: str) -> str:
    """搜索课程文档库，返回匹配的文档标题和摘要。

    当用户询问以下内容时调用此工具：
    - 课程/教程/文档相关内容（"Python入门文档"、"LangChain教程"、"怎么配置中间件"）
    - 技术学习资料（"有没有关于Agent开发的资料"）

    以下情况不应调用此工具：
    - 搜索互联网/外部网页 → 非课程文档范围
    - 查询天气、计算税费、汇率换算 → 非文档类问题
    - 执行代码或调试程序 → 需用代码执行工具

    返回字符串：匹配到的文档数量和标题摘要
    """
    print("调用工具：search_docs")
    return "找到 3 篇相关文档。"

qwen_model = init_chat_model(
    model="qwen3.7-max",
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY"),
)

deepseek_model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
)

# 从工具对象动态提取名称，避免硬编码导致的幻觉问题
tools = [get_weather, calculate_tax, search_docs]
tool_names = "、".join(t.name for t in tools)

tool_selector_prompt = (
    "你的任务是根据用户问题选择最相关的工具。\n"
    f"可用工具名只有：{tool_names}。\n"
    "必须从已有的工具名中选择，不允许创造、翻译、改写工具名。\n"
    '请严格返回 JSON object，格式为：{"tools": ["工具名"]}。'
)

agent = create_agent(
    model=qwen_model,
    tools=tools,
    middleware=[
        LLMToolSelectorMiddleware(
            model=deepseek_model,
            max_tools=2,
            system_prompt=tool_selector_prompt
        )
    ],
)

resp = agent.invoke(
    {"messages": [HumanMessage("帮我查一下文档，顺便查下北京的天气")]}
)

rprint(resp)

## 3.5 ToolRetryMiddleware
+ 作用： 当工具调用因临时性错误（如网络抖动、服务暂时不可用、超时等）失败时，自动进行重试，提高 Agent 工具执行的容错性和稳定性
+ 原理： 拦截工具执行过程中抛出的异常，根据重试配置等待一段时间后重新执行工具调用，直到成功或达到最大重试次数
+ 适用场景： 适合搜索、HTTP API、数据库查询等可能出现短暂失败的工具；不适合支付、下单、写库等非幂等工具，避免重复执行造成副作用
+ 失败处理： 达到最大重试次数后，可继续交给 Agent 处理，也可以直接抛出错误

**重点参数说明：**

| 参数 | 作用 | 常用写法 |
| --- | --- | --- |
| `max_retries` | 最大重试次数，不包含首次工具调用 | `max_retries=3` |
| `retry_on` | 指定哪些异常需要重试 | `retry_on=(TimeoutError,)` |
| `initial_delay` | 第一次重试前的等待时间，单位秒 | `initial_delay=0.3` |
| `backoff_factor` | 指数退避倍数，下一次等待时间按倍数增长 | `backoff_factor=2.0` |
| `max_delay` | 单次重试等待时间上限，防止等待过长 | `max_delay=2.0` |
| `jitter` | 是否加入随机抖动，避免大量请求同时重试 | `jitter=False` |
| `tools` | 限定只对指定工具启用重试；为 `None` 时作用于全部工具 | `tools=["search_docs"]` |
| `on_failure` | 所有重试失败后的处理方式 | `"continue"`、`"error"` |

**案例：观察重试过程和指数退避**

下面示例使用真实模型触发一次工具调用，工具 `flaky_search` 前 3 次都会生成错误信息并通过 `TimeoutError` 抛出，第 4 次才正常返回。  
每次工具执行时都会打印：

- 当前是第几次执行
- 距离上一次工具执行的实际间隔
- 按指数退避计算出的理论等待时间

这里设置 `jitter=False`，因此实际等待时间会接近 `0.3s -> 0.6s -> 1.2s`，可以清晰看到指数退避过程。

In [ ]:
import os
import time
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

load_dotenv(".env")

retry_demo_state = {"attempt": 0, "last_started_at": None}
initial_delay = 2.0
backoff_factor = 2.0


@tool
def flaky_search(query: str) -> str:
    """查询指定主题的课程资料。必须通过这个工具查询 ToolRetryMiddleware。"""
    now = time.perf_counter()
    retry_demo_state["attempt"] += 1
    attempt = retry_demo_state["attempt"]

    if retry_demo_state["last_started_at"] is None:
        print(f"attempt {attempt}: 第一次执行工具，无等待")
    else:
        actual_wait = now - retry_demo_state["last_started_at"]
        print(f"attempt {attempt}: 距离上次执行 {actual_wait:.3f}s，")

    retry_demo_state["last_started_at"] = now

    if attempt < 4:
        error_message = f"错误信息：第 {attempt} 次查询 {query} 时发生临时超时"
        print(f"attempt {attempt}: {error_message}，触发重试")
        raise TimeoutError(error_message)

    result = (
        f"查询成功：{query}。"
        f"本工具第 {attempt} 次调用成功，前 {attempt - 1} 次都因临时超时触发了 ToolRetryMiddleware 重试。"
    )
    print(f"attempt {attempt}: {result}")
    return result


model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)

agent = create_agent(
    model=model,
    tools=[flaky_search],
    middleware=[
        ToolRetryMiddleware(
            max_retries=3,
            retry_on=(TimeoutError,),
            initial_delay=initial_delay,  # 2.0秒
            backoff_factor=backoff_factor,  # 2.0秒
            max_delay=20.0,  # 20秒
            jitter=False,
            on_failure="error",
        )
    ],
)

resp = agent.invoke(
    {
        "messages": [
            HumanMessage(
                "请必须调用 flaky_search 工具查询 ToolRetryMiddleware。"
                "工具返回成功结果后，请根据工具返回内容用一句话总结重试过程。"
                "不要跳过工具调用，也不要直接根据常识回答。"
            )
        ]
    }
)
print(resp["messages"][-1].content)

## 3.6 LLMToolEmulator
+ 作用： 用 LLM 模拟工具执行结果，而不真正执行工具函数，适合在开发、测试、演示阶段替代慢工具、贵工具或依赖外部服务的工具
+ 原理： 拦截工具调用请求，读取工具名、工具描述和入参，再调用一个模拟模型生成 ToolMessage 作为工具结果；被模拟的工具函数本体不会执行
+ 适用场景： 适合测试 Agent 工具调用链路、演示工具效果、规避外部 API 不稳定或成本较高的问题
+ 注意事项： 模拟结果由模型生成，不代表真实工具结果；涉及支付、下单、写库等高风险工具时，只能用于测试环境

**重点参数说明：**

| 参数 | 作用 | 常用写法 |
| --- | --- | --- |
| `tools` | 指定需要被模拟的工具；为 `None` 时模拟全部工具；为空列表时不模拟任何工具 | `tools=["get_weather"]` |
| `model` | 用来生成模拟工具结果的模型；不传时会使用默认模型 | `model=emulator_model` |

In [ ]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolEmulator
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

load_dotenv(".env")


@tool
def get_weather(city: str) -> str:
    """查询指定城市的实时天气信息。"""
    return f"{city} 台风，28 度。"


model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)

emulator_model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)

agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[LLMToolEmulator(tools=["get_weather"], model=emulator_model)],
)

resp = agent.invoke({"messages": [HumanMessage("帮我查询杭州天气")]})
print(resp["messages"][-1].content)

## 3.7 ContextEditingMiddleware
+ 作用： 在上下文过长时自动裁剪历史工具结果，减少模型输入 token，避免旧工具输出占满上下文窗口
+ 原理： 在模型调用前检查消息列表的 token 数；超过触发阈值后，按配置将较早的 `ToolMessage` 内容替换为占位符，并保留最近若干条工具结果
+ 适用场景： 适合长对话、多轮工具调用、工具返回内容很长的 Agent；尤其适合搜索、日志分析、数据库查询等容易产生大段工具结果的任务
+ 注意事项： 该中间件会编辑传给模型的上下文，不会修改真实工具执行结果；如果后续推理依赖完整工具输出，需要调大 `keep` 或排除关键工具

**重点参数说明：**

| 参数 | 作用 | 常用写法 |
| --- | --- | --- |
| `edits` | 上下文编辑策略列表；默认使用 `ClearToolUsesEdit()` | `edits=[ClearToolUsesEdit(...)]` |
| `token_count_method` | token 统计方式；`approximate` 更快，`model` 更准确但依赖模型实现 | `token_count_method="approximate"` |
| `trigger` | 触发上下文编辑的 token 阈值 | `trigger=2000` |
| `keep` | 保留最近多少条工具结果不清理 | `keep=1` |
| `clear_at_least` | 至少需要清理出的 token 数；达到后停止继续清理 | `clear_at_least=500` |
| `clear_tool_inputs` | 是否同时清理 AIMessage 中原始工具调用参数 | `clear_tool_inputs=True` |
| `exclude_tools` | 指定不参与清理的工具名 | `exclude_tools=["audit_log"]` |
| `placeholder` | 被清理工具结果替换成的占位文本 | `placeholder="[工具结果已清理]"` |

**案例：多个工具结果清理前后的 token 消耗对比**

下面示例创建 4 个工具，并依次调用它们来构造历史工具结果。  
我们只关注 `ContextEditingMiddleware` 生效前后的输入 token 数：先打印编辑前 token，再打印编辑后 token 和节省 token。

In [ ]:
import os

from rich import print as rprint
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import (
    ClearToolUsesEdit,
    ContextEditingMiddleware,
)
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.tools import tool

load_dotenv(".env")


@tool
def search_docs(query: str) -> str:
    """搜索课程文档。"""
    return "文档检索结果：" + (
        "ContextEditingMiddleware 会在模型调用前清理旧工具结果。" * 120
    )


@tool
def query_database(sql: str) -> str:
    """查询课程数据库。"""
    return "数据库查询结果：" + (
        "user_id=1001, action=tool_call, status=success, created_at=2026-07-10。" * 100
    )


@tool
def fetch_service_logs(service: str) -> str:
    """读取服务日志。"""
    return "服务日志：" + (
        "WARN context window high; INFO tool output appended; DEBUG retry finished。"
        * 100
    )


@tool
def inspect_metrics(metric: str) -> str:
    """查询系统指标。"""
    return "指标结果：" + ("token_count=128000, latency_ms=320, cache_hit=true。" * 100)


model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)

agent = create_agent(
    model=model,
    tools=[search_docs, query_database, fetch_service_logs, inspect_metrics],
    middleware=[
        ContextEditingMiddleware(
            edits=[
                ClearToolUsesEdit(
                    trigger=300,
                    keep=1,
                    placeholder="[工具结果已清理]",
                )
            ],
            token_count_method="approximate",
        ),
    ],
)

messages = [
    HumanMessage(
        content="按步骤完成以下任务："
        "1. 搜索课程文档。"
        "2. 查询课程数据库。"
        "3. 读取服务日志。"
        "4. 查询系统指标。"
        "5. 输出结果。"
    )
]

# 使用中间件前后 toekn消耗 8666 -> 2803
rprint(agent.invoke({"messages": messages}))

# 4. 自定义中间件

## 4.1 hook 函数

hook 函数是插入到 Agent 执行流程中的回调函数，用来在关键阶段前后增加自定义逻辑，例如修改上下文、记录日志、限制调用、处理异常或改变执行流程。

常见 hook

| 类型 | hook 函数 | 关注点 | 典型用途 |
| --- | --- | --- | --- |
| Node-style hooks | before_agent | Agent 开始前 | 初始化状态、检查输入 |
| Node-style hooks | before_model | 模型调用前 | 裁剪上下文、注入提示词 |
| Node-style hooks | after_model | 模型返回后 | 检查输出、更新状态 |
| Node-style hooks | after_agent | Agent 结束后 | 记录结果、统一收尾 |
| Wrap-style hooks | wrap_model_call | 一次模型调用 | 重试、降级、缓存、模型路由 |
| Wrap-style hooks | wrap_tool_call | 一次工具调用 | 工具重试、异常兜底、结果改写 |
| 动态提示词 | dynamic_prompt | 系统提示词生成 | 根据上下文生成 prompt |


Node-style hooks

| 特点 | 说明 |
| --- | --- |
| 执行方式 | 作为 Agent 图中的独立节点运行 |
| 输入对象 | state、runtime |
| 返回结果 | 返回字典更新 Agent 状态；不修改则返回 None |
| 适合场景 | 状态更新、输入检查、上下文整理、日志记录、流程跳转 |

核心特点：在流程节点前后做处理，不直接控制模型或工具的真实调用过程。

Wrap-style hooks

| 特点 | 说明 |
| --- | --- |
| 执行方式 | 包裹一次模型调用或工具调用 |
| 输入对象 | request、handler |
| 控制能力 | 可修改请求、调用 handler、处理结果，也可跳过 handler 直接返回 |
| 适合场景 | 重试、缓存、模型降级、模型路由、工具异常兜底、结果改写 |

核心特点：控制一次具体调用的执行方式，控制力比 Node-style hooks 更强。


选择建议

| 需求 | 推荐 hook 类型 |
| --- | --- |
| 检查输入、整理上下文、更新状态 | Node-style hooks |
| 控制模型调用、切换模型、缓存结果 | Wrap-style hooks |
| 控制工具调用、处理工具异常、自动重试 | Wrap-style hooks |
| 根据运行上下文生成系统提示词 | dynamic_prompt |

一句话总结：Node-style hooks 处理流程节点，Wrap-style hooks 控制具体调用。

## 4.2 Node-style hooks

支持两种用法
+ 装饰器是**函数式挂载**，把一个hook快速挂载到Agent的某个节点
+ 类写法是**对象化中间件**，把中间件封装成一个可配置、可复用、可拓展的组件

### 1. 基于装饰器的实现

In [12]:
import os
from typing import Any
from langchain.agents.middleware import (
    Runtime,
    before_agent,
    before_model,
    after_agent,
    after_model,
)
from dotenv import load_dotenv
from langchain.agents import AgentState, create_agent
from langchain.chat_models import init_chat_model

load_dotenv(".env")

# 这里的hook函数就是中间件

@before_model
def before_model_middleware(
    state: AgentState, runtime: Runtime
) -> dict[str, Any] | None:
    state["messages"][-1].content += "-----> before_model <-----"
    return None


@after_model
def after_model_middleware(
    state: AgentState, runtime: Runtime
) -> dict[str, Any] | None:
    state["messages"][-1].content += "-----> after_model <-----"
    return None


@before_agent
def before_agent_middleware(
    state: AgentState, runtime: Runtime
) -> dict[str, Any] | None:
    state["messages"][-1].content += "-----> before_agent <-----"
    return None


@after_agent
def after_agent_middleware(
    state: AgentState, runtime: Runtime
) -> dict[str, Any] | None:
    state["messages"][-1].content += "-----> after_agent <-----"
    return None


model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)

agent = create_agent(
    model=model,
    middleware=[
        before_model_middleware,
        after_model_middleware,
        before_agent_middleware,
        after_agent_middleware,
    ],
)


resp = agent.invoke({"messages": [HumanMessage(content="你好，简要介绍以下自己")]})

for msg in resp["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好，简要介绍以下自己-----> before_agent <----------> before_model <-----
================================== Ai Message ==================================

你好！我是 DeepSeek，由深度求索公司创造的 AI 助手。我能回答问题、解释概念、辅助写作、写代码，也支持文件上传和处理。我很乐意随时为你提供帮助——有什么想聊的或需要解决的吗？😊-----> after_model <----------> after_agent <-----


### 2. 基于类的写法

**关键规则**：
+ 类必须继承 **AgentMiddleware**
+ 方法名固定 before_agent、after_agent、before_model、after_model

In [ ]:
import os
from typing import Any
from langchain.agents.middleware import AgentMiddleware
from dotenv import load_dotenv
from langchain.agents import AgentState, create_agent
from langchain.chat_models import init_chat_model


class MyMiddleware(AgentMiddleware):
    @before_agent
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "-----> before_agent <-----"
        return None

    @after_agent
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "-----> after_agent <-----"
        return None

    @before_model
    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "-----> before_model <-----"
        return None

    @after_model
    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "-----> after_model <-----"
        return None


model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)

agent = create_agent(
    model=model,
    middleware=[
        before_model_middleware,
        after_model_middleware,
        before_agent_middleware,
        after_agent_middleware,
    ],
)

resp = agent.invoke({"messages": [HumanMessage(content="你好，简要介绍以下自己")]})

for msg in resp["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好，简要介绍以下自己-----> before_agent <----------> before_model <-----
================================== Ai Message ==================================

你好！我是 DeepSeek，由深度求索公司创造的 AI 助手，专注于提供准确、友好的解答和帮助。有什么我可以为你做的吗？-----> after_model <----------> after_agent <-----
